# Technical Plan: Creating (Date, Lat, Lon) Tensors from NO2 and TEMP Data

## Overview
This notebook outlines the process to reshape CSV data into regular tensors with shape (dates, lats, lons).
Since NO2 and TEMP come from different datasets with different lat-lon grids, they will have different tensor shapes.

In [1]:
import pandas as pd
import numpy as np
from scipy.interpolate import griddata
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load and Explore Data

In [2]:
# Load both CSV files
no2_df = pd.read_csv('~/Downloads/NO2_manual_1213m.csv')
no2_df['date'] = pd.to_datetime(no2_df['timestamp'].str.strip()).dt.date

# TEMP is hourly - average to daily for consistent tensor shape
temp_df = pd.read_csv('~/Downloads/TEMP_manual_27929m.csv')
temp_df['date'] = pd.to_datetime(temp_df['timestamp'].str.strip()).dt.date
temp_df = temp_df.groupby(['date', 'latitude', 'longitude'])['TEMP'].mean().reset_index()
print(f"TEMP after daily averaging: {len(temp_df)} points")

print("NO2 Data Shape:", no2_df.shape)
print("NO2 Columns:", no2_df.columns.tolist())
print("\nTEMP Data Shape:", temp_df.shape)
print("TEMP Columns:", temp_df.columns.tolist())

TEMP after daily averaging: 480 points
NO2 Data Shape: (183754, 5)
NO2 Columns: ['timestamp', 'NO2', 'latitude', 'longitude', 'date']

TEMP Data Shape: (480, 4)
TEMP Columns: ['date', 'latitude', 'longitude', 'TEMP']


In [3]:
print(temp_df.head(10))

         date   latitude  longitude        TEMP
0  2019-01-01  28.278565  77.171222  287.244276
1  2019-01-01  28.278565  77.422121  286.832046
2  2019-01-01  28.278565  77.673019  286.059734
3  2019-01-01  28.278565  77.923917  285.778783
4  2019-01-01  28.529464  77.171222  286.054012
5  2019-01-01  28.529464  77.422121  286.168254
6  2019-01-01  28.529464  77.673019  286.403771
7  2019-01-01  28.529464  77.923917  286.668289
8  2019-01-01  28.780362  77.171222  285.941623
9  2019-01-01  28.780362  77.422121  286.049182


In [4]:
# Inspect unique dates and spatial extent
# no2_df['date'] = pd.to_datetime(no2_df['date'].str.strip()).dt.date
# temp_df['date'] = pd.to_datetime(temp_df['date'].str.strip()).dt.date

print("NO2 unique dates:", no2_df['date'].nunique())
print("NO2 lat range:", no2_df['latitude'].min(), "-", no2_df['latitude'].max())
print("NO2 lon range:", no2_df['longitude'].min(), "-", no2_df['longitude'].max())
print("NO2 unique lats:", no2_df['latitude'].nunique())
print("NO2 unique lons:", no2_df['longitude'].nunique())

print("\nTEMP unique dates:", temp_df['date'].nunique())
print("TEMP lat range:", temp_df['latitude'].min(), "-", temp_df['latitude'].max())
print("TEMP lon range:", temp_df['longitude'].min(), "-", temp_df['longitude'].max())
print("TEMP unique lats:", temp_df['latitude'].nunique())
print("TEMP unique lons:", temp_df['longitude'].nunique())

NO2 unique dates: 28
NO2 lat range: 28.19767841391827 - 29.18942510477315
NO2 lon range: 77.1511982864422 - 78.14294497729708
NO2 unique lats: 92
NO2 unique lons: 92

TEMP unique dates: 30
TEMP lat range: 28.27856528246453 - 29.03126022831689
TEMP lon range: 77.17122232386458 - 77.92391726971692
TEMP unique lats: 4
TEMP unique lons: 4


## Step 2: Create Regular Lat-Lon Grid

In [5]:
def create_regular_grid(df, value_col, n_lat=92, n_lon=92):
    """
    Create a regular (date, lat, lon) tensor from scattered data.
    
    Parameters:
    ----------
    df : DataFrame with columns: date, latitude, longitude, <value_col>
    value_col : Name of the value column (e.g., 'NO2' or 'TEMP')
    n_lat : Number of latitude grid points
    n_lon : Number of longitude grid points
    
    Returns:
    --------
    tensor : np.array of shape (n_dates, n_lat, n_lon)
    dates : Array of unique dates
    lat_grid : Regular latitude grid
    lon_grid : Regular longitude grid
    """
    # Get unique dates
    dates = sorted(df['date'].unique())
    n_dates = len(dates)
    
    # Create regular lat/lon grids - use sorted unique values from data for consistent shape
    lat_grid = np.sort(df['latitude'].unique())
    lon_grid = np.sort(df['longitude'].unique())
    n_lat = len(lat_grid)
    n_lon = len(lon_grid)
    
    # Initialize tensor with NaN
    tensor = np.full((n_dates, n_lat, n_lon), np.nan)
    
    # Fill tensor for each date
    for d_idx, date in enumerate(dates):
        date_data = df[df['date'] == date]
        
        if len(date_data) == 0:
            continue
        
        # Get existing points and values
        known_lats = date_data['latitude'].values
        known_lons = date_data['longitude'].values
        known_vals = date_data[value_col].fillna(np.nan).values
        
        # Only interpolate where we have valid values
        valid_mask = ~np.isnan(known_vals)
        if valid_mask.sum() < 3:  # Need at least 3 points for interpolation
            continue
        
        # Create meshgrid for interpolation
        lat_mesh, lon_mesh = np.meshgrid(lat_grid, lon_grid)
        points = np.column_stack([known_lats[valid_mask], known_lons[valid_mask]])
        values = known_vals[valid_mask]
        grid_points = np.column_stack([lat_mesh.ravel(), lon_mesh.ravel()])
        
        # Interpolate (linear for scattered data)
        interpolated = griddata(points, values, grid_points, method='linear')
        tensor[d_idx] = interpolated.reshape(n_lat, n_lon)
    
    return tensor, np.array(dates), lat_grid, lon_grid

# Use original unique lat/lon counts - no downsampling
no2_n_lat = no2_df['latitude'].nunique()
no2_n_lon = no2_df['longitude'].nunique()
temp_n_lat = temp_df['latitude'].nunique()
temp_n_lon = temp_df['longitude'].nunique()

print(f"NO2 grid: {no2_n_lat} x {no2_n_lon}")
print(f"TEMP grid: {temp_n_lat} x {temp_n_lon}")

NO2_tensor, no2_dates, no2_lat_grid, no2_lon_grid = create_regular_grid(
    no2_df, 'NO2', n_lat=no2_n_lat, n_lon=no2_n_lon
)
TEMP_tensor, temp_dates, temp_lat_grid, temp_lon_grid = create_regular_grid(
    temp_df, 'TEMP', n_lat=temp_n_lat, n_lon=temp_n_lon
)

print("NO2 Tensor Shape:", NO2_tensor.shape)
print("TEMP Tensor Shape:", TEMP_tensor.shape)

NO2 grid: 92 x 92
TEMP grid: 4 x 4
NO2 Tensor Shape: (28, 92, 92)
TEMP Tensor Shape: (30, 4, 4)


## Step 3: Temporal Interpolation

In [6]:
def temporal_interpolate(tensor, dates, target_freq='H'):
    """
    Interpolate tensor temporally to fill gaps.
    
    Parameters:
    ----------
    tensor : np.array of shape (n_dates, n_lat, n_lon)
    dates : Array of dates
    target_freq : Target frequency ('H' for hourly, 'D' for daily)
    
    Returns:
    --------
    interpolated_tensor :Tensor with more time steps
    new_dates : New date array
    """
    from scipy.interpolate import interp1d
    
    n_dates, n_lat, n_lon = tensor.shape
    
    # Simple linear interpolation along time axis
    # For each spatial point, interpolate in time
    interpolated = np.zeros_like(tensor)
    
    for i in range(n_lat):
        for j in range(n_lon):
            time_series = tensor[:, i, j]
            valid_mask = ~np.isnan(time_series)
            
            if valid_mask.sum() >= 2:
                f = interp1d(
                    np.where(valid_mask)[0], 
                    time_series[valid_mask],
                    kind='linear',
                    bounds_error=False,
                    fill_value=np.nan
                )
                interpolated[:, i, j] = f(np.arange(n_dates))
    
    return interpolated, dates

# Apply temporal interpolation
NO2_tensor_filled, _ = temporal_interpolate(NO2_tensor, no2_dates)
TEMP_tensor_filled, _ = temporal_interpolate(TEMP_tensor, temp_dates)

print("NO2 Filled Tensor Shape:", NO2_tensor_filled.shape)
print("TEMP Filled Tensor Shape:", TEMP_tensor_filled.shape)
print("\nNO2 NaN count before:", np.isnan(NO2_tensor).sum())
print("NO2 NaN count after:", np.isnan(NO2_tensor_filled).sum())
print()
print("\nTEMP NaN count before:", np.isnan(TEMP_tensor).sum())
print("NO2 NaN count after:", np.isnan(TEMP_tensor_filled).sum())

NO2 Filled Tensor Shape: (28, 92, 92)
TEMP Filled Tensor Shape: (30, 4, 4)

NO2 NaN count before: 27170
NO2 NaN count after: 6768


TEMP NaN count before: 0
NO2 NaN count after: 0


## Step 4: Summary

In [7]:
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"\nNO2:")
print(f"  - Original data points: {len(no2_df)}")
print(f"  - Unique dates: {len(no2_dates)}")
print(f"  - Final tensor shape: {NO2_tensor_filled.shape}")
print(f"  - Lat grid: {no2_lat_grid[0]:.4f} to {no2_lat_grid[-1]:.4f}")
print(f"  - Lon grid: {no2_lon_grid[0]:.4f} to {no2_lon_grid[-1]:.4f}")

print(f"TEMP:")
print(f"  - Original data points: {len(temp_df)}")
print(f"  - Unique dates: {len(temp_dates)}")
print(f"  - Final tensor shape: {TEMP_tensor_filled.shape}")
print(f"  - Lat grid: {temp_lat_grid[0]:.4f} to {temp_lat_grid[-1]:.4f}")
print(f"  - Lon grid: {temp_lon_grid[0]:.4f} to {temp_lon_grid[-1]:.4f}")

SUMMARY

NO2:
  - Original data points: 183754
  - Unique dates: 28
  - Final tensor shape: (28, 92, 92)
  - Lat grid: 28.1977 to 29.1894
  - Lon grid: 77.1512 to 78.1429
TEMP:
  - Original data points: 480
  - Unique dates: 30
  - Final tensor shape: (30, 4, 4)
  - Lat grid: 28.2786 to 29.0313
  - Lon grid: 77.1712 to 77.9239
